# MyPortfolio notebook

This notebook is our **workbench** for the MyPortfolio app. Each section tests one chunk of the app, broken into small steps you can run one at a time.

**How the project is split**

| Where | What lives there | Who uses it |
|---|---|---|
| `core/` folder | The real logic: fetching prices, saving portfolios, and later the math | This notebook *and* the Streamlit app |
| `MyPortfolio.ipynb` | Step-by-step tests and explanations of `core/` | Us, while building |
| Streamlit app | The screens people click on | Our partner builds this |

**How to use this notebook**

1. In VS Code, click **Select Kernel** (top right) and pick the **.venv** Python.
2. Click into a grey code cell and press **Shift + Enter** to run it.
3. Run cells from top to bottom the first time.

> MyPortfolio gives rules of thumb based on past prices. It is not financial advice.

## Setup

Run this cell first, every time you open the notebook.

In [ ]:
# %autoreload makes the notebook notice when a file in core/ is edited.
# Without it, you'd have to restart the notebook after every code change.
%load_ext autoreload
%autoreload 2

# Bring in our own code from the core/ folder.
from core import data                 # the price fetcher
from core import portfolio as pf      # the portfolio saver ("pf" is a short nickname)
from core import my_stocks as ms      # the MyStocks section ("ms" is a short nickname)
from core import financials as fin    # MyAnalysis Part 3A: statements and ratios
from core import rivals as rv         # MyAnalysis Part 3B: comparing with rivals
from core import economy as eco       # MyAnalysis Part 3C: the economy
from core import technicals as ta     # MyAnalysis Part 3D: the price chart
from core import analysts as an       # MyAnalysis Part 3E: the experts
from core import dcf                  # MyAnalysis Part 3F: the DCF valuation
from core import verdict as vd        # MyAnalysis Part 3G: the final verdict
from core import my_optimizer as mo   # MyOptimizer: the best mix for your rules

# Bring in helper libraries.
import pandas as pd                   # tables of numbers
import plotly.express as px           # quick charts

# Show money with 2 decimals in tables, so 342.940002 shows as 342.94.
pd.options.display.float_format = "{:,.2f}".format

print("Ready!")

---
# Chunk 1: Foundation

The foundation has two parts:

* **`core/data.py`** goes to Yahoo Finance and brings back prices and company facts.
* **`core/portfolio.py`** writes down what you own and saves it to a file.

Everything else in the app is built on top of these two.

### Step 1.1: Download daily prices

`data.get_prices()` takes a list of tickers and a time period. It gives back a table where each **row is a day** and each **column is a stock**.

The prices are *adjusted*. That means stock splits and dividends are already baked in, so comparing prices over time is fair.

In [ ]:
# The stocks we want to look at. Lowercase and extra spaces are fine,
# the code tidies them up for us.
tickers = ["AAPL", "MSFT", "SPY"]

# Download 6 months of daily prices.
prices = data.get_prices(tickers, period="6mo")

# .tail() shows the last 5 rows, i.e. the 5 most recent days.
prices.tail()

### Step 1.2: Chart the prices

A picture is easier than a table. Each line is one stock.

In [ ]:
# px.line draws one line per column of the table.
fig = px.line(prices, title="Daily closing prices", labels={"value": "Price ($)", "variable": "Ticker"})
fig.show()

### Step 1.3: It remembers answers

The second time you ask the same question, it answers instantly from memory instead of downloading again. Answers are kept for one hour.

`%time` measures how long a line takes.

In [ ]:
# Same question as Step 1.1, so this should take almost no time.
%time _ = data.get_prices(tickers, period="6mo")

### Step 1.4: Today's prices

`data.get_latest_prices()` gives just the most recent price for each stock.

In [ ]:
latest = data.get_latest_prices(tickers)
latest

### Step 1.5: Company facts

`data.get_info()` gets the fact sheet for one company. Here we loop over our tickers and stack the answers into one table.

| Fact | What it means |
|---|---|
| `market_cap` | What the whole company is worth, in dollars |
| `pe_ratio` | Price-to-earnings. How many dollars you pay for each $1 the company earned last year |
| `forward_pe` | Same, but using next year's *expected* earnings |
| `dividend_yield` | Yearly cash paid to owners divided by the price. 0.02 means 2% |
| `beta` | How much the stock swings compared to the market. 1 means the same as the market |

ETFs like SPY are baskets of stocks, so some facts come back blank (`None`). That's normal.

In [ ]:
# Ask for each company's facts, then turn the list of dictionaries into a table.
facts = pd.DataFrame([data.get_info(t) for t in tickers])

# Make the ticker the row label, then format each column for humans.
# .style.format only changes how numbers LOOK, not the numbers themselves.
facts.set_index("ticker").style.format(
    {
        "market_cap": "${:,.0f}",      # 5020252110848 -> $5,020,252,110,848
        "pe_ratio": "{:.1f}",          # 38.912895 -> 38.9
        "forward_pe": "{:.1f}",
        "dividend_yield": "{:.2%}",    # 0.0031 -> 0.31%
        "beta": "{:.2f}",
    },
    na_rep="n/a",                      # show blanks as "n/a"
)

### Step 1.6: Bad tickers don't crash anything

If one ticker is wrong, you get a pink warning box and the good tickers still come back.

In [ ]:
# "ZZZZFAKE" doesn't exist. Watch for the warning under this cell.
data.get_prices(["AAPL", "ZZZZFAKE"], period="5d").tail(2)

### Step 1.7: Build a practice portfolio

A portfolio is a dictionary with two lists:

* **holdings**: one row per purchase, like a stack of receipts
* **trades**: a diary of every buy, which the warnings sidebar will read later

`pf.add_holding()` checks your inputs, then adds a receipt and a diary line.

In [ ]:
# Start with an empty portfolio.
my_portfolio = pf.empty_portfolio()

# Add three purchases: ticker, shares, price paid per share, date bought.
pf.add_holding(my_portfolio, "AAPL", shares=10, buy_price=150.00, buy_date="2024-01-15")
pf.add_holding(my_portfolio, "MSFT", shares=2.5, buy_price=400.00, buy_date="2025-03-01")
pf.add_holding(my_portfolio, "SPY", shares=3, buy_price=550.00, buy_date="2025-06-10")

# Show the holdings as a table.
pf.holdings_table(my_portfolio)

In [ ]:
# The trade diary got a matching line for each purchase.
pf.trades_table(my_portfolio)

### Step 1.8: Save it, then load it back

We save to a **practice file** so we never touch a real portfolio while testing.

In [ ]:
# Where the practice portfolio goes. It's inside data/, which git ignores,
# so personal holdings never get uploaded.
practice_file = "data/practice_portfolio.json"

# Write it to the file.
pf.save_portfolio(my_portfolio, practice_file)

# Read it back from the file, as if we'd just reopened the app.
reloaded = pf.load_portfolio(practice_file)
pf.holdings_table(reloaded)

### Step 1.9: Mistakes get friendly messages

Bad inputs raise a `ValueError` with a plain-English message. In the Streamlit app, our partner can show that message straight to the user.

In [ ]:
# A list of mistakes a user might make: (ticker, shares, price, date)
mistakes = [
    ("", 10, 150, None),            # blank ticker
    ("AAPL", 0, 150, None),         # zero shares
    ("AAPL", 10, -5, None),         # negative price
    ("AAPL", "ten", 150, None),     # words instead of numbers
    ("AAPL", 10, 150, "15/01/2024"),# wrong date format
    ("AAPL", 10, 150, "2099-01-01"),# date in the future
]

# Try each one and print the message instead of crashing.
for ticker, shares, price, when in mistakes:
    try:
        pf.add_holding(pf.empty_portfolio(), ticker, shares, price, when)
    except ValueError as error:
        print("Caught:", error)

### Try it yourself

* Change `tickers` in Step 1.1 to stocks you like, then rerun Steps 1.1 to 1.5.
* Try `period="5y"` and look at the chart.
* Add one of your own purchases in Step 1.7.

### For our partner: plugging this into Streamlit

**Use `st.session_state` for the portfolio on the live app, not `save_portfolio()`.** On a hosted Streamlit app, the saved file is shared by every visitor and gets wiped whenever the app restarts. Session state gives each visitor their own portfolio.

```python
import streamlit as st
from core import data, portfolio as pf

# Give each visitor their own empty portfolio the first time they arrive.
if "portfolio" not in st.session_state:
    st.session_state.portfolio = pf.empty_portfolio()

# A small form to add a purchase.
with st.form("add_holding"):
    ticker = st.text_input("Ticker")
    shares = st.number_input("Shares", min_value=0.0)
    price = st.number_input("Buy price", min_value=0.0)
    when = st.date_input("Buy date")
    if st.form_submit_button("Add"):
        try:
            pf.add_holding(st.session_state.portfolio, ticker, shares, price, when)
        except ValueError as error:
            st.error(str(error))   # shows our friendly message in red

# Show what they own.
st.dataframe(pf.holdings_table(st.session_state.portfolio))
```

---
# Chunk 2: MyStocks

MyStocks answers **"What do I own, and am I winning?"** All of it lives in **`core/my_stocks.py`**, plus selling and fixing mistakes in **`core/portfolio.py`**.

Three ideas to know first:

* **Canadian and US stocks stay in separate groups.** Each group's dollars stay in its own currency, so exchange rate swings never blur how your stocks did. Only the pie chart converts, because it needs one currency to compare slice sizes.
* **You race an S&P 500 twin.** Every time you bought, an imaginary twin spent the same money on the S&P 500 (the SPY fund) on the same day. If your stocks grew more than the twin's, you beat the market.
* **Every number ending in `_pct` is a percent.** 5.0 means 5%.

### Step 2.1: A practice portfolio with Canadian and US stocks

Canadian stocks end in **.TO** (Toronto Stock Exchange). Buying the same stock twice at different prices is fine. Each purchase is its own receipt.

In [ ]:
# Start fresh so this section doesn't depend on Chunk 1.
practice = pf.empty_portfolio()

# US stocks. Two separate Apple purchases at different prices.
pf.add_holding(practice, "AAPL", shares=10, buy_price=150, buy_date="2024-01-15")
pf.add_holding(practice, "AAPL", shares=5, buy_price=200, buy_date="2025-01-15")
pf.add_holding(practice, "MSFT", shares=2, buy_price=400, buy_date="2025-03-01")

# Canadian stocks (prices in Canadian dollars).
pf.add_holding(practice, "RY.TO", shares=20, buy_price=130, buy_date="2024-06-03")    # Royal Bank
pf.add_holding(practice, "SHOP.TO", shares=10, buy_price=100, buy_date="2024-08-12")  # Shopify

pf.holdings_table(practice)

### Step 2.2: Sell some shares

`pf.sell_holding()` uses the **average cost** method, the same way Canada works out gains for taxes:

* average cost = total you paid / total shares
* gain = (sell price - average cost) x shares sold

Here we own 15 Apple shares with an average cost of $166.67. Selling 5 at $300 locks in (300 - 166.67) x 5 = **$666.67**.

In [ ]:
# Sell 5 of our 15 Apple shares at $300 each.
pf.sell_holding(practice, "AAPL", shares=5, sell_price=300, sell_date="2026-01-05")

# Each Apple receipt shrank by the same third, so the average cost didn't change.
pf.holdings_table(practice)

In [ ]:
# The trade diary now has a "sell" line with the gain we locked in.
pf.trades_table(practice)

### Step 2.3: One row per stock

`ms.positions_table()` combines the receipts so each stock gets one row, then adds every standard number.

| Column | What it means |
|---|---|
| `avg_cost` | Total paid divided by total shares |
| `value` | Shares x today's price. What it's worth right now |
| `gain` / `gain_pct` | How much you're up or down, in the stock's own currency and as a percent |
| `day_change` / `day_change_pct` | How much it moved since yesterday's close |
| `weight_pct` | This stock's slice of your whole pie. Uses today's exchange rate |

In [ ]:
positions = ms.positions_table(practice)

# Format the numbers for humans. This only changes how they LOOK.
positions.style.format(
    {
        "shares": "{:,.2f}",
        "avg_cost": "{:,.2f}", "cost": "{:,.2f}", "price": "{:,.2f}", "value": "{:,.2f}",
        "gain": "{:+,.2f}",                 # the + shows a plus sign on gains
        "gain_pct": "{:+.1f}%",
        "day_change": "{:+,.2f}",
        "day_change_pct": "{:+.2f}%",
        "weight_pct": "{:.1f}%",
        "pe_ratio": "{:.1f}",
        "dividend_yield_pct": "{:.2f}%",
    },
    na_rep="n/a",
)

### Step 2.4: Where your money is

The donut chart shows each stock's slice. To compare a Canadian stock with a US one, the slices use today's exchange rate.

In [ ]:
ms.allocation_chart(positions).show()

### Step 2.5: The headline numbers

`ms.portfolio_summary()` gives one set of numbers per currency group, plus everything combined in Canadian dollars.

**How to read "beat the S&P 500 by"**: it's your percent gain minus the twin's percent gain. Positive means you won. The comparison ignores exchange rates on purpose.

In [ ]:
summary = ms.portfolio_summary(practice)

# Print each currency group in plain English.
for currency, g in summary["groups"].items():
    print(f"--- {currency} stocks ({g['stocks']} holdings) ---")
    print(f"Worth now:          {g['value']:>12,.2f} {currency}")
    print(f"You paid:           {g['cost']:>12,.2f} {currency}")
    print(f"Gain:               {g['gain']:>+12,.2f} {currency}  ({g['gain_pct']:+.1f}%)")
    print(f"Today:              {g['day_change']:>+12,.2f} {currency}  ({g['day_change_pct']:+.2f}%)")
    print(f"Locked in by sales: {g['realized_gain']:>+12,.2f} {currency}")
    print(f"S&P 500 twin:       {g['sp500_twin_gain_pct']:>+11.1f}%")
    print(f"You beat it by:     {g['beat_sp500_by_pct']:>+11.1f} percentage points")
    print()

# The combined total, converted into Canadian dollars.
c = summary["combined"]
print(f"Everything together: {c['value']:,.2f} {c['currency']}")
print(f"Exchange rates used: {c['fx_rates']}")

### Step 2.6: You vs your twin over time

`ms.value_history()` builds a day-by-day value for one currency group, next to the twin. Both lines jump up when you buy, because both include the money you put in. **The gap between the lines is how far ahead of the market you are.**

The chart uses the shares you own **now**. Shares you sold are left out, as if you'd always held your current amount.

In [ ]:
# One chart per currency group.
for currency in summary["groups"]:
    history = ms.value_history(practice, currency)
    ms.value_chart(history, currency).show()

### Step 2.7: Fix a typo

`pf.remove_holding()` deletes a purchase by its row number in `holdings_table()`, starting from 0. Its diary line goes too, because a typo isn't a real trade. For real sales, use `sell_holding()` instead.

In [ ]:
# Oops, typed "APPL" instead of "AAPL". It lands in the last row.
pf.add_holding(practice, "APPL", shares=1, buy_price=100, buy_date="2025-01-02")
last_row = len(practice["holdings"]) - 1

# Delete it.
pf.remove_holding(practice, last_row)

# APPL is gone from both tables.
print(pf.holdings_table(practice)["ticker"].tolist())
print(pf.trades_table(practice)["ticker"].tolist())

### Try it yourself

* Swap in stocks you own in Step 2.1, then rerun Steps 2.1 to 2.6.
* Sell all of one stock in Step 2.2 and see it vanish from the table.
* Try selling more shares than you own, and read the error message.

### For our partner: plugging MyStocks into Streamlit

```python
import streamlit as st
from core import my_stocks as ms

portfolio = st.session_state.portfolio            # set up in Chunk 1's example

summary = ms.portfolio_summary(portfolio)

# One row of headline numbers per currency group.
for currency, g in summary["groups"].items():
    st.subheader(f"{currency} stocks")
    col1, col2, col3 = st.columns(3)
    col1.metric("Worth", f"{g['value']:,.2f}", f"{g['day_change_pct']:+.2f}% today")
    col2.metric("Gain", f"{g['gain']:,.2f}", f"{g['gain_pct']:+.1f}%")
    col3.metric("vs S&P 500", f"{g['beat_sp500_by_pct']:+.1f} pts")
    st.plotly_chart(ms.value_chart(ms.value_history(portfolio, currency), currency))

positions = ms.positions_table(portfolio)
st.dataframe(positions)
st.plotly_chart(ms.allocation_chart(positions))

# Selling: show our friendly message if something's wrong.
try:
    pf.sell_holding(portfolio, ticker, shares, price, when)
except ValueError as error:
    st.error(str(error))
```

---
# Chunk 3: MyAnalysis

MyAnalysis looks at any stock, whether you own it or not, from seven angles. Each part has its own file in `core/`.

| Part | Question it answers | File |
|---|---|---|
| 3A | Is the business healthy? | `financials.py` |
| 3B | Is it better than its rivals? | `rivals.py` |
| 3C | Is the economy helping or hurting? | `economy.py` |
| 3D | What's the price doing? | `technicals.py` |
| 3E | What are the experts saying? | `analysts.py` |
| 3F | What's the business really worth? | `dcf.py` |
| 3G | So what should I do? | `verdict.py` |

## Part 3A: Financial statements and ratios

Every public company publishes three **financial statements**:

| Statement | Think of it as | Tells you |
|---|---|---|
| Income statement | The yearly report card | Money in, money out, and profit |
| Balance sheet | A photo of one day | What it owns (assets) and what it owes (liabilities) |
| Cash flow statement | The bank account history | Where real cash came from and where it went |

**Ratios** turn those big numbers into small ones you can compare. "Apple made $112 billion" is hard to judge. "Apple keeps 27 cents of profit from every $1 of sales" is easy to compare with other companies.

**The columns you'll see:**

* **TTM** means *trailing twelve months*, the latest four quarters added up. Its price-based ratios use **today's** share price.
* **FY2025** means the company's business year that ended in 2025. Its price-based ratios use the share price on the **last day of that year**. Business years don't always end in December. Apple's ends in September.

### Step 3A.1: The income statement

`fin.statements()` gives all three statements with just the key lines. `fin.format_statement()` makes the numbers readable: **B** means billions and **M** means millions.

In [ ]:
# Pick any company. Canadian ones end in .TO
company = "AAPL"

# Get all three statements. This is a dictionary with three tables inside.
apple_statements = fin.statements(company)

# Show the income statement with readable numbers.
fin.format_statement(apple_statements["income"])

### Step 3A.2: The balance sheet and cash flow statement

In [ ]:
# What it owns and owes, on the last day of each year.
fin.format_statement(apple_statements["balance"])

In [ ]:
# Where the cash went. Negative numbers are cash going OUT,
# like capital spending, dividends, and buying back shares.
fin.format_statement(apple_statements["cashflow"])

### Step 3A.3: Every ratio, every year

`fin.ratios_table()` calculates 23 ratios in five families: valuation, profitability, debt and safety, growth, and cash.

The **trend** column compares the newest business year with the oldest one shown:

* **improving** means it moved in the good direction. For valuation ratios like P/E, "improving" means the stock got *cheaper* compared with its profits.
* **worsening** means it moved in the bad direction.
* **steady** means it changed less than 5%.
* **n/a** means there isn't enough data, or neither direction is clearly better.

In [ ]:
# Calculate everything. This keeps the real numbers for later math.
apple_ratios = fin.ratios_table(company)

# Show a readable copy, grouped by family.
fin.format_ratios(apple_ratios)

### Step 3A.4: What does each ratio mean?

`fin.ratio_guide()` explains every ratio and shows its formula. Keep this handy while reading the table above.

In [ ]:
# Let long text wrap instead of being cut off.
pd.set_option("display.max_colwidth", None)
fin.ratio_guide()

### Step 3A.5: Chart the trends

`fin.ratio_chart()` draws chosen ratios over the years, oldest on the left and TTM on the right. Pick ratios with the same unit so they share one scale.

In [ ]:
# Three profit margins on one chart: all percents, so they share a scale.
fin.ratio_chart(
    apple_ratios,
    ["gross_margin_pct", "operating_margin_pct", "net_margin_pct"],
    title=f"{company} profit margins",
).show()

In [ ]:
# Returns on money: equity, assets, and invested capital.
fin.ratio_chart(
    apple_ratios,
    ["roe_pct", "roa_pct", "roic_pct"],
    title=f"{company} returns",
).show()

### Step 3A.6: Banks look different

Banks don't sell products, so they have no "gross profit" or "current assets". Their cash flow also mixes in customer deposits and loans. So for banks and insurers:

* Some ratios show **n/a**. That's expected, not a bug.
* Cash flow ratios are left blank on purpose, and you'll see a warning saying so.
* Debt to equity looks huge, around 4x. That's normal for a bank, because customer deposits count as money it owes.

In [ ]:
# Royal Bank of Canada. Watch for the pink warning box.
fin.format_ratios(fin.ratios_table("RY.TO"))

### Step 3A.7: Grab one number

The table from `ratios_table()` keeps real numbers, so you can pull any one out with `.loc[ratio, period]`. Later parts, like the final verdict, read numbers this way.

In [ ]:
pe_now = apple_ratios.loc["pe_ratio", "TTM"]
margin_trend = apple_ratios.loc["net_margin_pct", "trend"]

print(f"{company} costs {pe_now:.1f}x its last 12 months of profit.")
print(f"Its net profit margin is {margin_trend}.")

### Try it yourself

* Change `company` in Step 3A.1 to `"NVDA"` and rerun Steps 3A.1 to 3A.5. Look at how fast its margins grew.
* Try `"SHOP.TO"`. It trades in Canadian dollars but reports in US dollars, and a warning explains how that's handled.
* Try `"SPY"`. Funds have no financial statements, so you'll get a friendly warning instead of a table.

### For our partner: plugging Part 3A into Streamlit

```python
import warnings
import streamlit as st
from core import financials as fin

ticker = st.text_input("Ticker", "AAPL")

# Catch our warnings so they show on screen instead of the terminal.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ratios = fin.ratios_table(ticker)
    statements = fin.statements(ticker)
for w in caught:
    st.warning(str(w.message))

st.dataframe(fin.format_ratios(ratios))

# One tab per statement.
tab1, tab2, tab3 = st.tabs(["Income", "Balance sheet", "Cash flow"])
tab1.dataframe(fin.format_statement(statements["income"]))
tab2.dataframe(fin.format_statement(statements["balance"]))
tab3.dataframe(fin.format_statement(statements["cashflow"]))

st.plotly_chart(fin.ratio_chart(ratios, ["gross_margin_pct", "net_margin_pct"]))

with st.expander("What do these ratios mean?"):
    st.dataframe(fin.ratio_guide())
```

## Part 3B: Comparing with rivals

A ratio means more next to a rival's. A 30% profit margin is amazing for a grocery store and weak for a software company. Part 3B finds a company's **rivals**, lines up their ratios, and scores four qualities from **0 to 100**:

| Quality | What it asks | How it's measured |
|---|---|---|
| Management quality | Does management treat shareholders' money well? | Returns rising over the years, buying back shares, keeping debt in check, insiders owning stock |
| Competitive advantage | Does it earn more than rivals, year after year? | Operating margin now, operating margin in its worst year, return on invested capital |
| Brand strength | Can it charge more than rivals? | Gross margin now, in its worst year, and whether it's rising |
| Industry trends | Is the whole industry doing well? | Share of companies growing sales, improving margins, and beating the S&P 500 |

**How to read a score:** 80 means the company beats 80% of its rivals on those measures. **67 or more is strong, 34 to 66 is average, and below 34 is weak.** Industry trends works differently: 80 means about 80% of the whole industry is growing, improving, and beating the market.

> **These are stand-ins.** Nobody can measure management quality or brand strength exactly from numbers. For example, Costco keeps its markup low on purpose and earns money from memberships, so "pricing power" scores it as weak on brand strength. Always read the **why** next to each score.

### Step 3B.1: Find the rivals

`rv.find_rivals()` starts with the biggest companies in the same industry. It skips tiny ones and skips the company itself, even when it's listed in another country. If the industry has fewer than three, it tops up from the broader sector.

In [ ]:
company = "NVDA"
rv.find_rivals(company)

### Step 3B.2: Compare and score

`rv.compare_to_rivals()` does everything in one go. **The first run takes 15 to 30 seconds** because it downloads data for every rival. After that it's fast for an hour.

It hands back a dictionary with five things inside: the rivals, a side-by-side table, the scores, the details behind them, and industry facts.

In [ ]:
result = rv.compare_to_rivals(company)

print("Compared with:", result["rivals"])
result["scores"]

In [ ]:
# The same scores as a chart.
rv.scores_chart(result["scores"], company).show()

### Step 3B.3: Side by side

One row per company, using the latest numbers: the last 12 months where available, otherwise the newest business year. The bottom row is the **median** rival, the one right in the middle.

In [ ]:
result["side_by_side"].style.format(
    {
        "P/E": "{:.1f}",
        "Gross margin %": "{:.1f}%",
        "Operating margin %": "{:.1f}%",
        "Net margin %": "{:.1f}%",
        "Return on equity %": "{:.1f}%",
        "Return on invested capital %": "{:.1f}%",
        "Debt / equity": "{:.2f}",
        "Revenue growth %": "{:+.1f}%",
        "1-year stock return %": "{:+.1f}%",
    },
    na_rep="n/a",
)

### Step 3B.4: Chart any column

`rv.rivals_chart()` draws one column as bars. The company is blue, and the dashed line is the median rival.

In [ ]:
rv.rivals_chart(result["side_by_side"], "Operating margin %").show()

In [ ]:
rv.rivals_chart(result["side_by_side"], "Return on invested capital %").show()

### Step 3B.5: What's behind each score

Every score is the average of a few measurements. This table shows the company's number, the median rival's number, and what share of rivals it beats.

* **"Change in ..."** compares the newest business year with the oldest. Measured in **points**, so a margin going from 20% to 25% is +5 points.
* **"in its worst year"** asks: even in its worst year, how good was it? Staying strong in a bad year is a sign of a lasting advantage.
* **Share count below 0** means the company bought back its own shares, which gives each remaining share a bigger slice.
* Tiny differences in insider ownership and share count, under 1 point, count as a tie.

In [ ]:
result["details"].style.format(
    {"company": "{:,.2f}", "rivals_median": "{:,.2f}", "beats_pct": "{:.0f}%"},
    na_rep="n/a",
)

### Step 3B.6: The industry as a whole

In [ ]:
facts = result["industry"]
print(facts["why"])
print(f"Typical sales growth in the group:     {facts['median_revenue_growth_pct']:+.1f}%")
print(f"Typical 1-year stock return:           {facts['median_1y_return_pct']:+.1f}%")
print(f"S&P 500 over the same year:            {facts['sp500_1y_return_pct']:+.1f}%")

### Step 3B.7: Pick your own rivals

Yahoo's automatic list isn't always right. Apple's industry is "consumer electronics", which is mostly tiny companies, so the automatic list fills up with chip makers from the tech sector. You can pass your own list instead.

In [ ]:
apple_vs_big_tech = rv.compare_to_rivals("AAPL", rivals=["MSFT", "GOOGL", "SONO", "DELL"])
apple_vs_big_tech["scores"]

### Try it yourself

* Try `"RY.TO"`. Banks get scored on net margin and return on equity, and brand strength shows n/a.
* Try `"COST"` and read the "why" for brand strength. It shows how a stand-in can misread a business model.
* Try `"SHOP.TO"` against your own list of rivals.

### For our partner: plugging Part 3B into Streamlit

```python
import streamlit as st
from core import rivals as rv

ticker = st.text_input("Ticker", "NVDA")
custom = st.text_input("Your own rivals, comma separated (optional)")
rivals = [t.strip() for t in custom.split(",") if t.strip()] or None

# The first run is slow, so show a spinner.
with st.spinner("Comparing with rivals..."):
    try:
        result = rv.compare_to_rivals(ticker, rivals=rivals)
    except ValueError as error:
        st.error(str(error))
        st.stop()

st.plotly_chart(rv.scores_chart(result["scores"], ticker))
for quality, row in result["scores"].iterrows():
    st.markdown(f"**{quality}: {row['verdict']}**. {row['why']}")

st.dataframe(result["side_by_side"])
column = st.selectbox("Chart", list(result["side_by_side"].columns[1:]))
st.plotly_chart(rv.rivals_chart(result["side_by_side"], column))
```

## Part 3C: The economy

Even a great company swims against the current when the economy turns against it. Part 3C reads three big numbers about the **US economy** and works out whether each one is a **headwind** (holding a stock back) or a **tailwind** (pushing it along).

**Why only the US?** Most Canadian investors hold lots of US stocks, and the US 10-year yield is the world's benchmark interest rate. Canada's rates and economy usually move the same way.

| Number | What it is | Where it comes from |
|---|---|---|
| Interest rates | The US 10-year government bond yield: what the US pays to borrow for 10 years | Yahoo Finance |
| Inflation | How much higher prices are than a year ago. Central banks aim for **2%** | US Bureau of Labor Statistics |
| Economic growth | GDP growth: how fast the whole economy grows, averaged over the past year | US Bureau of Economic Analysis |

**How a headwind or tailwind is worked out:**

1. Each number gets a direction. Rates are rising, steady, or falling. Inflation is above, near, or below target. Growth is strong, normal, weak, or shrinking.
2. Each sector has a rule of thumb for how it usually reacts. Banks earn more when rates rise. Real estate and utilities usually suffer.
3. The company's own numbers can make the rates effect bigger. Lots of debt, or a high P/E, makes rising rates hurt more.

> **These are rules of thumb.** They describe what usually happens, not what will happen.

### Step 3C.1: The economy right now

`eco.snapshot()` gives each number now and a year ago, with a plain-English label. The **change** is in percentage points: 3.96% to 5.15% is +1.18 points.

In [ ]:
economy_now = eco.snapshot()

economy_now.style.format(
    {"now": "{:.2f}%", "a_year_ago": "{:.2f}%", "change": "{:+.2f}"},
    na_rep="n/a",
)

### Step 3C.2: Five years of history

All three numbers on one chart. The dashed line is the 2% inflation target. GDP growth jumps around a lot from quarter to quarter, which is why the label uses the average of the last four.

In [ ]:
eco.economy_chart().show()

### Step 3C.3: The rules of thumb

How each sector usually reacts to each number. Read across a row: technology usually gets hurt when rates rise and helped when growth is strong.

In [ ]:
eco.sensitivity_table()

### Step 3C.4: Headwinds and tailwinds for one stock

`eco.economy_effects()` combines the economy, the sector's rules of thumb, and the company's own debt and P/E.

* **sector_effect** comes from the rule of thumb: +1 is a push up, -1 is a push down.
* **company_effect** is the extra push from lots of debt or a high P/E. It only applies to interest rates.
* **The score** runs from 0 to 100, and **50 means neutral**. 60 or more is "helping", 40 or less is "hurting", in between is "mixed".

NextEra Energy is a utility with a lot of debt, so it's a good example.

In [ ]:
utility = eco.economy_effects("NEE")
print(f"{utility['company']} ({utility['sector']}): economy score {utility['score']:.0f}, {utility['verdict']}")

pd.set_option("display.max_colwidth", None)
utility["effects"][["reading", "sector_effect", "company_effect", "total", "verdict", "why"]]

### Step 3C.5: Compare a few stocks

Same economy, very different effects, depending on the sector.

In [ ]:
rows = []
for ticker in ["RY.TO", "NVDA", "XOM", "NEE", "COST"]:
    result = eco.economy_effects(ticker)
    verdicts = result["effects"]["verdict"]
    rows.append({
        "ticker": ticker,
        "sector": result["sector"],
        "rates": verdicts["Interest rates"],
        "inflation": verdicts["Inflation"],
        "growth": verdicts["Economic growth"],
        "score": round(result["score"]),
        "overall": result["verdict"],
    })

pd.DataFrame(rows).set_index("ticker")

### Try it yourself

* Try a home builder like `"DHI"` (Consumer Cyclical). It's hurt by rising rates and high inflation.
* Try a fund like `"SPY"`. It has no sector, so it gets the whole-market average.
* Find a company with lots of debt in Part 3A, then check whether Part 3C gives it an extra push from rates.

### For our partner: plugging Part 3C into Streamlit

```python
import streamlit as st
from core import economy as eco

# The economy, the same for every stock.
st.subheader("The US economy")
now = eco.snapshot()
col1, col2, col3 = st.columns(3)
for col, topic in zip([col1, col2, col3], eco.TOPICS):
    row = now.loc[topic]
    col.metric(topic, f"{row['now']:.2f}%", f"{row['change']:+.2f} pts in a year", delta_color="off")
    col.caption(row["label"])
st.plotly_chart(eco.economy_chart())

# What it means for one stock.
ticker = st.text_input("Ticker", "NEE")
result = eco.economy_effects(ticker)
st.metric("Economy score", f"{result['score']:.0f} / 100", result["verdict"], delta_color="off")
for topic, row in result["effects"].iterrows():
    icon = {"tailwind": "🟢", "headwind": "🔴", "neutral": "⚪"}[row["verdict"]]
    st.markdown(f"{icon} **{topic}: {row['verdict']}**. {row['why']}")
```

## Part 3D: The price chart

Parts 3A to 3C ask "is this a good business?" Part 3D asks a different question: **what are buyers and sellers doing with the stock right now?** This is called **technical analysis**.

| Tool | In plain words |
|---|---|
| Moving average | The average closing price over the last 50 or 200 days. It smooths out daily noise so you can see the trend. |
| Golden cross | The 50-day average crosses **above** the 200-day: a classic good sign. A **death cross** is the opposite. |
| RSI | Relative strength index, 0 to 100. **Above 70**: ran up too fast, may cool off. **Below 30**: fell too fast, may bounce. |
| Support | A **floor**: a price where the stock kept bouncing up. |
| Resistance | A **ceiling**: a price where it kept getting pushed down. A broken ceiling often becomes a floor. |
| Breakout | Closing **above a ceiling** on heavier trading than usual. A **breakdown** is closing below a floor. |
| Entry stop | A "buy only if" price just above the ceiling, so you only get in if a breakout really happens. |
| Protective stop | A "get me out" price below the floor. It caps how much you can lose. |
| ATR | Average true range: how much the price normally moves in a day. Used as a safety cushion for stops. |
| Retracement | A pullback after a big run. Pullbacks often pause at **38.2%, 50% or 61.8%** of the run (Fibonacci levels). |
| Reward to risk | Distance up to the next ceiling divided by distance down to the stop. **2 or more** is usually seen as good. |
| Momentum | Returns over 3, 6 and 12 months compared with the S&P 500. Winners tend to keep winning for a while. |

> **These are suggested price levels, not trade orders.** Technical signals are educated guesses about crowd behaviour, and they're often wrong over short periods.

### Step 3D.1: Read the chart

`ta.technical_summary()` checks six signals. Each one scores **+1** (good sign), **0** (neutral) or **-1** (warning sign), and they're averaged into a score from 0 to 100, where **50 is neutral**.

In [ ]:
company = "NVDA"
tech = ta.technical_summary(company)

print(f"{company} closed at {tech['price']:.2f} on {tech['as_of']}.")
print(f"Technical score: {tech['score']:.0f} / 100, {tech['verdict']}.")

pd.set_option("display.max_colwidth", None)
tech["signals"]

### Step 3D.2: The chart

Hover over anything to see its value, and drag across the chart to zoom in. Double-click to zoom back out.

* **Candles**: green when the day closed higher than it opened, red when lower. The thin lines show the day's high and low.
* **Green dashed lines** are floors, **red dashed lines** are ceilings. The number in brackets is how many times the price touched it.
* **Grey dotted lines** are the Fibonacci retracement levels.
* **Blue** is the entry stop, **orange** is the protective stop.
* **Middle panel**: how many shares traded each day. **Bottom panel**: RSI, with the 30 and 70 lines.

In [ ]:
ta.technical_chart(tech).show()

### Step 3D.3: The trade setup

The suggested entry stop, protective stop, and reward to risk, each with the reason it was placed there.

In [ ]:
setup = tech["setup"]
print(f"Price now:           {setup['price']:.2f}")
print(f"Normal daily move:   {setup['daily_move_atr']:.2f}  (ATR)")
print()
print(f"Entry stop:          {setup['entry_stop']:.2f}")
print(f"   why: {setup['entry_why']}")
print(f"Protective stop:     {setup['protective_stop']:.2f}  ({setup['stop_distance_pct']:.1f}% below today's price)")
print(f"   why: {setup['stop_why']}")
print()
print(f"Nearest floor:       {setup['next_floor']:.2f}")
print(f"Next ceiling:        {setup['next_ceiling']:.2f}")
print(f"Reward to risk:      {setup['reward_to_risk']:.1f} to 1")

### Step 3D.4: Every floor and ceiling

All the levels found in the past year, highest first. More **touches** means a stronger level.

In [ ]:
tech["levels"]

### Step 3D.5: Retracements

The past year's big move, and the Fibonacci levels where a pullback (or a bounce) might pause.

In [ ]:
fib = tech["retracements"]
print(f"The big move was {fib['direction']}: from {fib['low']:.2f} ({fib['low_day']}) to {fib['high']:.2f} ({fib['high_day']}).")
print(fib["where"])
for level, level_price in fib["levels"].items():
    print(f"   {level} level: {level_price:.2f}")

### Step 3D.6: Momentum

Returns include dividends. The **difference** is the stock's return minus the S&P 500's, in percentage points.

In [ ]:
tech["momentum"].style.format(
    {"stock_pct": "{:+.1f}%", "sp500_pct": "{:+.1f}%", "difference": "{:+.1f}"},
    na_rep="n/a",
)

### Step 3D.7: Compare a few stocks

In [ ]:
rows = []
for ticker in ["NVDA", "AAPL", "RY.TO", "XOM", "SHOP.TO"]:
    t = ta.technical_summary(ticker)
    readings = t["signals"]["reading"]
    rows.append({
        "ticker": ticker,
        "trend": readings["Trend"],
        "RSI": readings["RSI"],
        "momentum": readings["Momentum"],
        "reward to risk": readings["Reward to risk"],
        "score": round(t["score"]),
        "overall": t["verdict"],
    })

pd.DataFrame(rows).set_index("ticker")

### Try it yourself

* Try a stock that's been falling and look for a **death cross** in the Trend row.
* Try a stock you own and check how far its protective stop is from today's price.
* Try `"SKHY"`, which only started trading recently. Signals that need more history say so instead of guessing.

### For our partner: plugging Part 3D into Streamlit

```python
import streamlit as st
from core import technicals as ta

ticker = st.text_input("Ticker", "NVDA")
tech = ta.technical_summary(ticker)

st.metric("Technical score", f"{tech['score']:.0f} / 100", tech["verdict"], delta_color="off")
st.plotly_chart(ta.technical_chart(tech), use_container_width=True)

for signal, row in tech["signals"].iterrows():
    icon = {1: "🟢", 0: "⚪", -1: "🔴"}[row["points"]]
    st.markdown(f"{icon} **{signal}: {row['reading']}**. {row['why']}")

setup = tech["setup"]
col1, col2, col3 = st.columns(3)
col1.metric("Entry stop", f"{setup['entry_stop']:.2f}")
col2.metric("Protective stop", f"{setup['protective_stop']:.2f}", f"-{setup['stop_distance_pct']:.1f}%")
col3.metric("Reward to risk", f"{setup['reward_to_risk']:.1f} : 1")
st.caption("Suggested price levels, not trade orders.")
```

## Part 3E: What the experts say

Professional **analysts** at banks and research firms study companies full time. They publish three things:

| What they publish | What it means |
|---|---|
| A **rating** | Strong buy, buy, hold, sell, or strong sell. We turn these into an **average from 1 to 5**, where 1 is strong buy and 3 is hold. |
| A **price target** | Where they think the price will be in a year. |
| A **profit forecast** | How much they think the company will earn per share (EPS) this quarter, next quarter, this year and next year. |

**Analyst momentum** asks which way their opinions are *moving*. When analysts keep raising forecasts and targets, the price often keeps rising for a while. Four checks:

1. Are profit forecasts for this year and next **higher than 90 days ago**?
2. In the last 30 days, did **more analysts raise** their forecast than cut it?
3. In the last 90 days, were there **more upgrades and raised targets** than downgrades and cuts?
4. Is the **share of "buy" ratings** higher than a few months ago?

> **Analysts lean optimistic.** "Sell" ratings are rare, partly because their firms often want to do business with the companies they cover. So an average near "hold" is a lukewarm sign, not a neutral one.
>
> **Canadian stocks** have fewer analysts, and individual price targets often come from US analysts covering the New York listing, so they may be in **US dollars**. Momentum only uses whether a target went up or down, so the currency doesn't affect it.

### Step 3E.1: The summary

`an.analyst_summary()` checks six signals. Two are about **consensus** (what analysts think now) and four are about **momentum** (which way they're moving). Each is +1, 0 or -1.

* **momentum_score** uses just the four momentum signals. The final verdict will combine it with the price momentum from Part 3D.
* **score** uses all six. Both run from 0 to 100, where **50 is neutral**.

In [ ]:
company = "NVDA"
experts = an.analyst_summary(company)

print(f"{company}: analyst score {experts['score']:.0f} / 100, {experts['verdict']}.")
print(f"Analyst momentum score: {experts['momentum_score']:.0f} / 100.")

pd.set_option("display.max_colwidth", None)
experts["signals"]

### Step 3E.2: The ratings breakdown

How many analysts say what, now and over the past few months. **buy_pct** is the share saying buy or strong buy. **average** is the 1 to 5 score.

In [ ]:
experts["ratings"].style.format(
    {"strong buy": "{:.0f}", "buy": "{:.0f}", "hold": "{:.0f}", "sell": "{:.0f}",
     "strong sell": "{:.0f}", "total": "{:.0f}", "buy_pct": "{:.0f}%", "average": "{:.2f}"}
)

In [ ]:
an.ratings_chart(experts["ratings"], company).show()

### Step 3E.3: Price targets

Where analysts think the price will be in a year. The **median** is the target right in the middle of the pack. Unlike the **mean** (the plain average), it isn't pulled around by one very high or very low target.

In [ ]:
targets = experts["targets"]
print(f"Today's price:  {targets['price']:.2f}")
for name in ["low", "median", "mean", "high"]:
    print(f"{name.capitalize():>7} target: {targets[name]:.2f}  ({targets[name + '_upside_pct']:+.1f}% from today)")

In [ ]:
an.targets_chart(experts).show()

### Step 3E.4: Recent rating and target changes

Every change in the last 90 days, newest first.

* **upgrade / downgrade**: the analyst changed their rating.
* **kept rating / repeated rating**: same rating, often with a new target.
* **started covering**: a new analyst began following the stock.
* **target_move**: whether the price target was raised, lowered, kept, or set for the first time.

Firms use different words for the same idea: "Overweight" and "Outperform" both mean roughly "buy".

In [ ]:
if experts["target_currency_note"]:
    print(experts["target_currency_note"])

experts["changes"].style.format({"target_from": "{:.2f}", "target_to": "{:.2f}"}, na_rep="")

### Step 3E.5: Profit forecasts

The average forecast of earnings per share, now vs 30 and 90 days ago. **raised_30d** and **cut_30d** count how many analysts moved their forecast up or down in the last month.

In [ ]:
experts["forecasts"].style.format(
    {"now": "{:.2f}", "30_days_ago": "{:.2f}", "90_days_ago": "{:.2f}",
     "change_90d_pct": "{:+.1f}%", "raised_30d": "{:.0f}", "cut_30d": "{:.0f}"},
    na_rep="n/a",
)

### Step 3E.6: Compare a few stocks

In [ ]:
rows = []
for ticker in ["NVDA", "AAPL", "RY.TO", "SHOP.TO", "CNQ.TO"]:
    e = an.analyst_summary(ticker)
    readings = e["signals"]["reading"]
    rows.append({
        "ticker": ticker,
        "average rating": readings["Average rating"],
        "target upside": readings["Price target upside"],
        "forecasts (90 days)": readings["Forecast changes (90 days)"],
        "momentum score": round(e["momentum_score"]),
        "score": round(e["score"]),
        "overall": e["verdict"],
    })

pd.DataFrame(rows).set_index("ticker")

### Try it yourself

* Try `"AAPL"`. The price is above the average target, so analysts see it as fully priced, even though most still rate it a buy.
* Try `"CNQ.TO"`. Analysts are lukewarm on the rating but keep raising their profit forecasts, which shows why momentum is checked separately.
* Try `"SPY"`. Funds have no analyst coverage, so the scores show as blank.

### For our partner: plugging Part 3E into Streamlit

```python
import streamlit as st
from core import analysts as an

ticker = st.text_input("Ticker", "NVDA")
experts = an.analyst_summary(ticker)

if experts["verdict"] == "no coverage":
    st.info("No analysts cover this stock.")
    st.stop()

col1, col2 = st.columns(2)
col1.metric("Analyst score", f"{experts['score']:.0f} / 100", experts["verdict"], delta_color="off")
col2.metric("Analyst momentum", f"{experts['momentum_score']:.0f} / 100")

for signal, row in experts["signals"].iterrows():
    icon = {1: "🟢", 0: "⚪", -1: "🔴"}[row["points"]]
    st.markdown(f"{icon} **{signal}: {row['reading']}**. {row['why']}")

st.plotly_chart(an.targets_chart(experts))
st.plotly_chart(an.ratings_chart(experts["ratings"], ticker))

with st.expander("Recent rating and target changes"):
    if experts["target_currency_note"]:
        st.caption(experts["target_currency_note"])
    st.dataframe(experts["changes"])
```

## Part 3F: What the business is really worth (DCF)

A **DCF**, or discounted cash flow model, values a business in one idea: **it's worth all the cash it will ever hand back, shrunk down because a dollar in the future is worth less than a dollar today.**

The steps:

1. Start from the cash the business made over the **last 12 months**.
2. Grow it for **10 years**: years 1 to 5 at a growth rate, then years 6 to 10 slowing down step by step to a long-term rate (2.5% by default).
3. After year 10, assume it grows at that long-term rate **forever**. The value of "forever" is called the **terminal value**.
4. **Discount** every future dollar back to today. A higher discount rate means future money is worth less today.
5. Add it up, adjust for debt and cash, divide by the number of shares, and compare with today's price.

| Word | In plain words |
|---|---|
| Free cash flow | Cash left after running the business and paying for new equipment |
| Cost of equity | What shareholders expect to earn: the US 10-year yield plus extra for risk |
| Beta | How much the stock swings compared with the market. Raw betas jump around, so like Bloomberg we pull them a third of the way toward 1 |
| Equity risk premium | The extra return investors want for owning stocks instead of government bonds. We use 5% |
| WACC | Weighted average cost of capital: the cost of equity and the cost of debt, weighted by how much of the company is shares vs debt |

**Banks and insurers** are valued on their **dividends** instead, discounted at the cost of equity. Their cash flow mixes in customer deposits and loans, so free cash flow means nothing for them. This is called a dividend discount model.

> **A DCF is only as good as its guesses.** Small changes to growth or the discount rate move the answer a lot, which is why every result comes with a sensitivity table. With the 10-year yield near 5%, many popular stocks come out below their price: future cash is worth less today when safe bonds pay that much.

### Step 3F.1: Value one stock

`dcf.dcf_valuation()` runs the whole model. Coca-Cola is a good first example: a steady business with predictable cash.

* **upside** is how far the DCF value is above (or below) today's price.
* The **verdict** is "undervalued" at 15% or more above the price, "overvalued" at 15% or more below, and "fairly valued" in between.
* The **score** runs from 0 to 100, where **50 means the value equals the price**.

In [ ]:
company = "KO"
valuation = dcf.dcf_valuation(company)

print(f"{company}: worth {valuation['value_per_share']:.2f} {valuation['currency']} a share by this DCF.")
print(f"Today's price is {valuation['price']:.2f}, so the upside is {valuation['upside_pct']:+.1f}%: {valuation['verdict']}.")
print(f"Score: {valuation['score']:.0f} / 100. Method: {valuation['method']}.")
for note in valuation["notes"]:
    print("Note:", note)

### Step 3F.2: Every assumption, and where it comes from

Each input, its value, and where it came from. **Every one of these can be changed** (Step 3F.6).

The growth rate for years 1 to 5 starts from **analysts' forecast of next year's earnings growth**, kept between 0% and 20%. Very fast growth rarely lasts five years.

In [ ]:
pd.set_option("display.max_colwidth", None)
valuation["assumptions"]

### Step 3F.3: The 10 years of cash

Each year's cash, how fast it grew, and what it's worth today. The **discount factor** is what $1 received that year is worth today: 0.70 means 70 cents.

In [ ]:
valuation["projection"].style.format(
    {"growth_pct": "{:.1f}%", "cash_flow": fin.format_money,
     "discount_factor": "{:.3f}", "present_value": fin.format_money}
)

### Step 3F.4: How the value is built

Each bar adds or takes away a piece of the value of one share. **Everything after year 10** is usually the biggest piece, which is why the long-term guesses matter so much. The dashed line is today's price.

In [ ]:
print(f"{valuation['bridge']['terminal_share_pct']:.0f}% of the value comes from after year 10.")
dcf.bridge_chart(valuation).show()

### Step 3F.5: What if the guesses are off?

The value per share for other discount rates (rows) and long-term growth rates (columns). The middle square is the main answer. **Green squares are worth more than today's price, red squares less.** Blank squares are where the discount rate is too close to growth for the math to work.

In [ ]:
dcf.sensitivity_chart(valuation).show()

### Step 3F.6: Use your own guesses

Pass in any of `growth_pct`, `terminal_growth_pct`, `discount_rate_pct` or `equity_risk_premium_pct`, all in percent. The assumptions table then says "your own choice" next to them.

In [ ]:
standard = dcf.dcf_valuation("AAPL")
my_guess = dcf.dcf_valuation("AAPL", growth_pct=10, terminal_growth_pct=3, discount_rate_pct=8)

print(f"Apple with the standard guesses: {standard['value_per_share']:.2f}")
print(f"Apple with my guesses:           {my_guess['value_per_share']:.2f}")
print(f"Today's price:                   {standard['price']:.2f}")

### Step 3F.7: Compare a few stocks

In [ ]:
rows = []
for ticker in ["KO", "AAPL", "NVDA", "RY.TO"]:
    v = dcf.dcf_valuation(ticker)
    rows.append({
        "ticker": ticker,
        "method": v["method"],
        "price": round(v["price"], 2),
        "DCF value": round(v["value_per_share"], 2),
        "upside %": round(v["upside_pct"], 1),
        "discount rate %": round(v["inputs"]["discount_rate_pct"], 2),
        "verdict": v["verdict"],
    })

pd.DataFrame(rows).set_index("ticker")

### Step 3F.8: When a DCF can't run

A DCF needs positive cash to grow. A company that's losing cash, or a fund with no financial statements, gets a plain-English message instead.

In [ ]:
for ticker in ["RIVN", "SPY"]:
    try:
        dcf.dcf_valuation(ticker)
    except ValueError as error:
        print("Caught:", error)

### Try it yourself

* Change `growth_pct` in Step 3F.6 until Apple's value matches today's price. That shows how much growth the market is expecting.
* Try `"RY.TO"` and read the notes about why dividend models come out low for banks.
* Try `equity_risk_premium_pct=4` on any stock and watch how much the value moves.

### For our partner: plugging Part 3F into Streamlit

```python
import streamlit as st
from core import dcf

ticker = st.text_input("Ticker", "KO")

# Sliders let people try their own guesses. None means "use the standard guess".
use_own = st.checkbox("Use my own guesses")
growth = st.slider("Growth, years 1 to 5 (%)", 0.0, 30.0, 8.0) if use_own else None
terminal = st.slider("Growth after year 10 (%)", 0.0, 4.0, 2.5)

try:
    valuation = dcf.dcf_valuation(ticker, growth_pct=growth, terminal_growth_pct=terminal)
except ValueError as error:
    st.error(str(error))
    st.stop()

col1, col2, col3 = st.columns(3)
col1.metric("DCF value", f"{valuation['value_per_share']:.2f}")
col2.metric("Today's price", f"{valuation['price']:.2f}")
col3.metric("Upside", f"{valuation['upside_pct']:+.1f}%", valuation["verdict"], delta_color="off")
for note in valuation["notes"]:
    st.caption(note)

st.plotly_chart(dcf.bridge_chart(valuation))
st.plotly_chart(dcf.sensitivity_chart(valuation))
with st.expander("Assumptions"):
    st.dataframe(valuation["assumptions"])
```

## Part 3G: The final verdict

Part 3G runs every MyAnalysis part on one stock, turns each into a score from 0 to 100, and blends them into **one overall score** and a plain-English **suggestion**, with the reasons behind it. It can also review **every stock you own** and flag the ones worth a closer look.

| Part score | Comes from | Starting weight |
|---|---|---|
| Business health | 3A: how many basic health checks it passes | 20% |
| Versus rivals | 3B: the average of its four quality scores | 15% |
| Value | 3F: how far the DCF value is from today's price | 15% |
| Economy | 3C: headwinds and tailwinds | 10% |
| Price chart | 3D: the technical score | 20% |
| Analysts | 3E: the analyst score, including analyst momentum | 20% |

| Overall score | Suggestion |
|---|---|
| 65 or more | looks strong |
| 50 to 65 | hold |
| 35 to 50 | look closer |
| below 35 | consider trimming |

* **Momentum** gets its own line: price momentum from 3D next to analyst momentum from 3E, averaged.
* **Strengths** come from the highest-scoring parts and **concerns** from the lowest-scoring ones, so the reasons point at what drove the verdict.
* If a part **can't run** (a fund has no DCF, a small company has no analysts), it's skipped and the other weights grow to fill the gap.

> **This is a rule of thumb, not financial advice.** It blends simple measures of past data. Use it to decide what to look into, not what to buy or sell.

### Step 3G.1: The verdict on one stock

`vd.stock_verdict()` runs everything. **The first run for a stock takes about half a minute**, mostly for the rivals part. Repeats within the hour are fast.

In [ ]:
company = "KO"
final = vd.stock_verdict(company)

print(f"{final['name']}: {final['score']:.0f} / 100, {final['verdict'].upper()}")
m = final["momentum"]
print(f"Momentum: {m['reading']} (price {m['price_pct']:.0f}/100, analysts {m['analyst_pct']:.0f}/100)")
print()
print("Strengths:")
for reason in final["strengths"]:
    print("  +", reason)
print("Concerns:")
for reason in final["concerns"]:
    print("  -", reason)

### Step 3G.2: The six part scores

Each part's score, the weight it got, and a one-line summary. Green bars are good (60 or more), red bars are concerns (40 or less), and the dashed line is the overall score.

In [ ]:
final["parts"].style.format({"score": "{:.0f}", "weight_pct": "{:.0f}%"}, na_rep="n/a")

In [ ]:
vd.verdict_chart(final).show()

### Step 3G.3: The health checks behind "Business health"

Simple yes/no checks on the Part 3A ratios. Banks and insurers get checks that make sense for them, like return on assets instead of free cash flow. Checks with no data are skipped, not failed.

In [ ]:
final["health_checks"]

### Step 3G.4: Use your own weights

Pass `weights` with any parts you want to change. Parts you leave out keep their starting weight, and **0 turns a part off**. Here, value matters a lot and the economy is ignored.

In [ ]:
value_investor = vd.stock_verdict("KO", weights={"Value": 40, "Economy": 0})
print(f"Starting weights: {final['score']:.0f} / 100, {final['verdict']}")
print(f"Value-focused:    {value_investor['score']:.0f} / 100, {value_investor['verdict']}")
value_investor["parts"][["score", "weight_pct"]].style.format({"score": "{:.0f}", "weight_pct": "{:.0f}%"}, na_rep="n/a")

### Step 3G.5: When parts can't run

A fund like SPY has no financial statements and no analysts, so only the economy and price chart parts can run. The **skipped_because** column says why each part was skipped.

In [ ]:
fund = vd.stock_verdict("SPY")
print(f"SPY: {fund['score']:.0f} / 100, {fund['verdict']}")
fund["parts"][["score", "weight_pct", "skipped_because"]].style.format({"score": "{:.0f}", "weight_pct": "{:.0f}%"}, na_rep="n/a")

### Step 3G.6: Review a whole portfolio

`vd.portfolio_review()` runs the verdict on every stock you own, **most worrying first**. A stock is **flagged** when its verdict is "look closer" or "consider trimming".

This uses the practice portfolio from Chunk 2. It takes a little while the first time, about half a minute per stock. Pass `include_rivals=False` to make it much faster.

In [ ]:
review = vd.portfolio_review(practice)
review["table"].style.format({"score": "{:.0f}"}, na_rep="n/a")

### Step 3G.7: Dig into one holding

`review["results"]` keeps every stock's full verdict, so you can look closer without running it again.

In [ ]:
holding = review["results"]["SHOP.TO"]
print(f"{holding['name']}: {holding['score']:.0f} / 100, {holding['verdict']}")
for reason in holding["concerns"]:
    print("  -", reason)
vd.verdict_chart(holding).show()

### Try it yourself

* Try `weights={"Price chart": 50, "Analysts": 50, "Business health": 0, "Versus rivals": 0, "Value": 0, "Economy": 0}` for a pure momentum view.
* Add a stock you own to the practice portfolio in Step 2.1 and rerun the review.
* Compare `include_rivals=True` and `False` on the same stock to see how much the rivals part moves the score.

### For our partner: plugging Part 3G into Streamlit

```python
import streamlit as st
from core import verdict as vd

ticker = st.text_input("Ticker", "KO")
with st.spinner("Running every part of MyAnalysis..."):
    final = vd.stock_verdict(ticker)

st.metric("Overall", f"{final['score']:.0f} / 100", final["verdict"], delta_color="off")
st.caption(f"Momentum: {final['momentum']['reading']}")
st.plotly_chart(vd.verdict_chart(final))

col1, col2 = st.columns(2)
col1.subheader("Strengths")
for reason in final["strengths"]:
    col1.markdown(f"- {reason}")
col2.subheader("Concerns")
for reason in final["concerns"]:
    col2.markdown(f"- {reason}")
st.caption("A rule of thumb based on past data, not financial advice.")

# The portfolio review, using the visitor's portfolio from Chunk 1's example.
if st.button("Review my whole portfolio"):
    with st.spinner("Reviewing every holding..."):
        review = vd.portfolio_review(st.session_state.portfolio, include_rivals=False)
    st.dataframe(review["table"])
```

---
# Chunk 4: MyOptimizer

MyOptimizer answers **"Which mix of stocks best fits my rules?"** You pick the stocks and set the rules. It tries 20,000 different mixes on the past three years of prices, throws out every mix that breaks your rules, fine-tunes the best one left, and shows it next to your current mix. All of it lives in **`core/my_optimizer.py`**.

| Rule | What it means | Starting value |
|---|---|---|
| `goal` | What "best" means: `"sharpe"` (highest Sharpe ratio), `"low_risk"` (smallest swings), or `"high_return"` | `"sharpe"` |
| `max_drawdown_pct` | The biggest drop from a peak you can stomach | 25 |
| `min_return_pct` | The least you'll accept per year | 8 |
| `max_weight_pct` | The most in any one stock | 30 |
| `min_weight_pct` | The least in any one stock. Above 0 forces it to keep every stock | 0 |
| `years` | How many years of past prices to test on | 3 |

| Word | In plain words |
|---|---|
| Yearly return | How fast the money grew per year, on average |
| Yearly swings | How much it bounced around ("volatility"). Bigger means bumpier |
| Sharpe ratio | (yearly return minus what a safe 3-month US bill pays) divided by yearly swings: **return per bump**. Above 1 is good, above 2 is excellent |
| Worst drop | The biggest fall from a peak to a low ("max drawdown"). -25% means $10,000 shrank to $7,500 at its worst |

Each stock's returns are measured in **its own currency**, the same choice as MyStocks.

**MyEdit** lets you take the best mix and tweak it by hand (Steps 4.8 to 4.10). You see your edit next to your current mix and the best mix, and it tells you if your edit breaks any of your rules.

> **It's based on the past.** The best mix for the last three years is rarely the best for the next three. Optimizers love whatever did best recently, which is why the "most in one stock" rule matters. Use the answer as a starting point, not an instruction.

### Step 4.1: Optimize a list of stocks

`mo.optimize()` takes a list of tickers and, optionally, your current mix as percentages, so the two can be compared.

In [ ]:
stocks = ["AAPL", "KO", "RY.TO", "SHOP.TO", "MSFT"]
my_mix = {"AAPL": 30, "KO": 10, "RY.TO": 35, "SHOP.TO": 20, "MSFT": 5}    # % of my money in each

result = mo.optimize(stocks, current=my_mix)

print(result["message"])
print(f"Tested on {result['period']['start']} to {result['period']['end']} ({result['period']['years']} years).")
print(f"Safe 3-month US bill pays {result['risk_free_pct']:.2f}%.")

### Step 4.2: Your mix vs the best mix

In [ ]:
result["comparison"].style.format(
    {"yearly return %": "{:.1f}%", "yearly swings %": "{:.1f}%", "Sharpe ratio": "{:.2f}",
     "worst drop %": "{:.1f}%", "$10,000 became": "${:,.0f}"}
)

In [ ]:
# How much money goes in each stock, now vs best. "change" is in percentage points.
result["weights"]

In [ ]:
mo.weights_chart(result).show()

### Step 4.3: Every mix tried

Each dot is one mix. **Up and to the left is good**: more return for less bumpiness. Coloured dots follow your rules (brighter means a better Sharpe ratio) and grey dots break them. The star is your current mix and the diamond is the best one.

In [ ]:
mo.cloud_chart(result).show()

### Step 4.4: How $10,000 would have grown

The best mix for the **Sharpe ratio** doesn't always end with the most money. It wins on smoothness: a smaller worst drop and fewer bumps along the way. To chase the most money instead, use `goal="high_return"` (Step 4.5).

In [ ]:
mo.growth_chart(result).show()

### Step 4.5: Change the goal and the rules

In [ ]:
rows = {}
for goal in ["sharpe", "low_risk", "high_return"]:
    r = mo.optimize(stocks, goal=goal)
    rows[mo.GOALS[goal]] = {**r["comparison"].loc["Best mix"], **r["best_weights"]}

pd.DataFrame(rows).T.style.format("{:.1f}")

In [ ]:
# Keep every stock (at least 10% each) and allow up to 40% in one.
balanced = mo.optimize(stocks, current=my_mix, min_weight_pct=10, max_weight_pct=40)
balanced["weights"]

### Step 4.6: When the rules can't all be met

If no mix follows every rule, nothing crashes. The `status` says "no mix fits" and the `message` names the hardest rule and how close you can get.

In [ ]:
too_greedy = mo.optimize(stocks, min_return_pct=60)
print(too_greedy["status"])
print(too_greedy["message"])

### Step 4.7: Start from your own portfolio

`mo.optimize_portfolio()` uses what you own (from MyStocks) as the current mix. `add` lets you try stocks you don't own yet. Here it's the practice portfolio from Chunk 2, plus two new ideas.

In [ ]:
mine = mo.optimize_portfolio(practice, add=["KO", "TD.TO"])
print(mine["message"])
mine["weights"]

### Step 4.8: MyEdit, tweak the best mix by hand

`mo.my_edit()` starts from the best mix and changes **only the stocks you name**. The others grow or shrink **in proportion**, so the total stays 100%.

Here, Coca-Cola goes from 30% down to 20%. The other four share the extra 10 points in the same proportions they had.

In [ ]:
edited = mo.my_edit(result, {"KO": 20})

print(edited["message"])
for problem in edited["breaks"]:
    print("  -", problem)

edited["weights"]

**Watch out:** sharing out the extra 10 points pushed Royal Bank above the 30% limit, so MyEdit flags it. MyEdit **lets you break your rules on purpose**, but always tells you when you do.

### Step 4.9: Your edit next to the other mixes

Every table and chart now includes **"Your edit"**. In the cloud chart it's the orange circle.

In [ ]:
edited["comparison"].style.format(
    {"yearly return %": "{:.1f}%", "yearly swings %": "{:.1f}%", "Sharpe ratio": "{:.2f}",
     "worst drop %": "{:.1f}%", "$10,000 became": "${:,.0f}"}
)

In [ ]:
mo.weights_chart(edited).show()
mo.cloud_chart(edited).show()
mo.growth_chart(edited).show()

### Step 4.10: More ways to edit

* **Edit an edit:** pass an edited result back in, and it starts from your last edit.
* **Set every stock yourself:** name them all. They must then add up to 100%.
* **Build one from scratch:** if no mix fit your rules, MyEdit starts from your current mix (or an equal split) instead.

In [ ]:
# Edit the edit: also give Royal Bank exactly 30%, so it's back within the limit.
fixed = mo.my_edit(edited, {"KO": 20, "RY.TO": 30})
print("Edit of an edit:", fixed["edit_weights"], "|", fixed["message"])

# Set every stock yourself: an equal split.
equal = mo.my_edit(result, {"AAPL": 20, "KO": 20, "RY.TO": 20, "SHOP.TO": 20, "MSFT": 20})
print("Equal split:   ", equal["edit_weights"], "|", equal["message"])

# Mistakes get plain-English messages.
for changes in [{"KO": 70, "AAPL": 40}, {"NVDA": 10}]:
    try:
        mo.my_edit(result, changes)
    except ValueError as error:
        print("Caught:", error)

### Try it yourself

* Add a few stocks you've been watching to `stocks` in Step 4.1.
* In Step 4.8, try `{"SHOP.TO": 25}` and see how much the Sharpe ratio drops.
* Set `max_drawdown_pct=15` and see how the best mix changes when you can't stomach big drops.
* Try `years=5` for a longer test. Stocks that haven't traded that long shorten the test for everyone, and a warning says so if it's under a year.

### For our partner: plugging MyOptimizer into Streamlit

```python
import streamlit as st
from core import my_optimizer as mo

goal = st.selectbox("Goal", list(mo.GOALS), format_func=mo.GOALS.get)
max_drop = st.slider("Biggest drop I can stomach (%)", 5, 60, 25)
min_return = st.slider("Minimum yearly return (%)", 0, 30, 8)
max_weight = st.slider("Most in one stock (%)", 10, 100, 30)
extra = st.text_input("Other stocks to consider, comma separated")
add = [t.strip() for t in extra.split(",") if t.strip()]

try:
    result = mo.optimize_portfolio(st.session_state.portfolio, add=add, goal=goal,
                                   max_drawdown_pct=max_drop, min_return_pct=min_return,
                                   max_weight_pct=max_weight)
except ValueError as error:
    st.error(str(error))
    st.stop()

if result["status"] != "ok":
    st.warning(result["message"])
else:
    st.success(result["message"])

# MyEdit: one number box per stock, starting at the best mix (or current if none fit).
st.subheader("MyEdit")
start = result["best_weights"] or result["current_weights"]
changes = {}
for ticker in result["tickers"]:
    new = st.number_input(f"{ticker} %", 0.0, 100.0, float(start.get(ticker, 0)), step=1.0)
    if abs(new - start.get(ticker, 0)) > 0.05:      # only send the ones you changed
        changes[ticker] = new
if changes:
    try:
        result = mo.my_edit(result, changes)
        st.info(result["message"])
        for problem in result["breaks"]:
            st.warning(problem)
    except ValueError as error:
        st.error(str(error))

st.dataframe(result["comparison"])
st.plotly_chart(mo.weights_chart(result))
st.plotly_chart(mo.cloud_chart(result))
st.plotly_chart(mo.growth_chart(result))
st.caption("Based on past prices. The best mix for the past is rarely the best for the future.")
```

---
# Coming next

Each chunk gets its own section here once it's approved and built.

5. **MyRisks**: concentration, sector, correlation, swings, and worst drops
6. **Warnings sidebar**: nudges about loss aversion and other traps when you trade